In [ ]:
import json
import os
import re
import sys
from pathlib import Path
from urllib.parse import quote

import requests
import brainunit as u

os.environ.setdefault("JAX_PLATFORMS", "cpu")


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "develop_doc").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import braincell
from braincell import Morpho

print("braincell version:", braincell.__version__)
print("repo_root:", repo_root)
            


# `neuromorpho_diff`: batch cache NeuroMorpho neurons, then batch compare with `braincell`

This notebook is organized as two direct steps:

1. use `q` and `fq` to batch query NeuroMorpho neurons
2. cache every neuron under `develop_doc/data/neuromorpho/<neuron_id>/`
3. store `metadata.json` with neuron metadata, measurement, and download links
4. scan cached folders, load standardized `SWC`, and compare against NeuroMorpho measurement


In [ ]:
API_BASE = "https://neuromorpho.org/api"
FILE_BASE = "https://neuromorpho.org/dableFiles"
session = requests.Session()

EXACT_MATCH_KEYS = ("n_stems", "n_branch", "n_bifs")
FLOAT_THRESHOLDS = {
    "length": 0.001,
    "surface": 0.01,
    "volume": 0.01,
    "pathDistance": 0.001,
    "eucDistance": 0.001,
    "soma_Surface": 0.10,
}


def fetch_neurons(*, q, fq=None, size=20, page=0, sort="neuron_id,asc"):
    params = {
        "q": q,
        "size": size,
        "page": page,
        "sort": sort,
    }
    if fq:
        params["fq"] = list(fq)

    response = session.get(f"{API_BASE}/neuron/select/", params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    neurons = payload.get("_embedded", {}).get("neuronResources", [])
    return neurons, payload.get("page", {}), response.url


def fetch_neuron_batch(*, q, fq=None, size=20, page_start=0, max_pages=1, sort="neuron_id,asc"):
    all_neurons = []
    query_urls = []
    seen_ids = set()

    for page in range(page_start, page_start + max_pages):
        neurons, page_info, query_url = fetch_neurons(q=q, fq=fq, size=size, page=page, sort=sort)
        query_urls.append(query_url)
        if not neurons:
            break

        for neuron in neurons:
            neuron_id = neuron["neuron_id"]
            if neuron_id in seen_ids:
                continue
            seen_ids.add(neuron_id)
            all_neurons.append(neuron)

        total_pages = page_info.get("totalPages")
        if total_pages is not None and page + 1 >= total_pages:
            break

    return all_neurons, query_urls


def measurement_url(neuron):
    return neuron["_links"]["measurements"]["href"]


def fetch_measurement(neuron):
    response = session.get(measurement_url(neuron), timeout=30)
    response.raise_for_status()
    return response.json()


def safe_filename(name):
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    return cleaned.strip("._") or "neuromorpho_neuron"


def standardized_swc_url(neuron):
    archive = quote(neuron["archive"].lower(), safe="")
    neuron_name = quote(neuron["neuron_name"], safe="")
    return f"{FILE_BASE}/{archive}/CNG%20version/{neuron_name}.CNG.swc"


def original_file_extension(neuron):
    original_format = neuron.get("original_format")
    if not original_format:
        raise ValueError("This neuron does not expose an original_format field.")

    suffix = Path(original_format).suffix
    if not suffix:
        raise ValueError(f"Cannot infer original file extension from original_format={original_format!r}")
    return suffix


def original_file_url(neuron):
    archive = quote(neuron["archive"].lower(), safe="")
    suffix = original_file_extension(neuron)
    filename = quote(f"{neuron['neuron_name']}{suffix}", safe="")
    return f"{FILE_BASE}/{archive}/Source-Version/{filename}"


def file_plan(neuron, *, mode):
    if mode not in {"standard", "original", "both"}:
        raise ValueError("mode must be one of: 'standard', 'original', 'both'")

    plans = []
    stem = safe_filename(neuron["neuron_name"])

    if mode in {"standard", "both"}:
        plans.append(
            {
                "kind": "standard",
                "url": standardized_swc_url(neuron),
                "filename": f"{stem}.CNG.swc",
            }
        )

    if mode in {"original", "both"}:
        try:
            suffix = original_file_extension(neuron)
        except ValueError as exc:
            print(f"skip original for neuron_id={neuron['neuron_id']}: {exc}")
        else:
            plans.append(
                {
                    "kind": "original",
                    "url": original_file_url(neuron),
                    "filename": f"{stem}{suffix}",
                }
            )

    return plans


def download_file(url, path, *, overwrite=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        return False

    with session.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with path.open("wb") as fh:
            for chunk in response.iter_content(chunk_size=1 << 15):
                if chunk:
                    fh.write(chunk)
    return True


def relative_to_repo(path):
    path = Path(path).resolve()
    try:
        return str(path.relative_to(repo_root))
    except ValueError:
        return str(path)


def repo_path(path_value):
    path = Path(path_value)
    return path if path.is_absolute() else repo_root / path


def neuron_cache_dir(output_root, neuron):
    return Path(output_root) / str(neuron["neuron_id"])


def cache_neuron_bundle(neuron, output_root, *, mode="standard", overwrite=False):
    folder = neuron_cache_dir(output_root, neuron)
    folder.mkdir(parents=True, exist_ok=True)

    measurement = fetch_measurement(neuron)
    download_items = []
    for plan in file_plan(neuron, mode=mode):
        path = folder / plan["filename"]
        downloaded_now = download_file(plan["url"], path, overwrite=overwrite)
        download_items.append(
            {
                "kind": plan["kind"],
                "url": plan["url"],
                "filename": path.name,
                "path": relative_to_repo(path),
                "downloaded_now": downloaded_now,
            }
        )

    metadata = {
        "neuron_id": neuron["neuron_id"],
        "neuron_name": neuron["neuron_name"],
        "archive": neuron.get("archive"),
        "original_format": neuron.get("original_format"),
        "measurement_url": measurement_url(neuron),
        "links": neuron.get("_links", {}),
        "neuron": neuron,
        "measurement": measurement,
        "download_mode": mode,
        "download_items": download_items,
    }

    metadata_path = folder / "metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8")
    return {
        "folder": folder,
        "metadata_path": metadata_path,
        "download_items": download_items,
        "measurement": measurement,
    }


def load_cached_metadata(folder):
    return json.loads((Path(folder) / "metadata.json").read_text(encoding="utf-8"))


def find_standard_swc(folder, metadata):
    for item in metadata.get("download_items", []):
        if item.get("kind") == "standard":
            path = repo_path(item["path"])
            if path.exists():
                return path

    swc_paths = sorted(Path(folder).glob("*.swc"))
    return swc_paths[0] if swc_paths else None


def compute_braincell_metrics(tree):
    soma_branches = [branch for branch in tree.branches if branch.type == "soma"]
    non_soma_branches = [branch for branch in tree.branches if branch.type != "soma"]
    return {
        "n_stems": int(tree.n_stems),
        "n_branch": int(len(non_soma_branches)),
        "n_bifs": int(sum(branch.n_children >= 2 for branch in non_soma_branches)),
        "length": float(sum(branch.length.to_decimal(u.um) for branch in non_soma_branches)),
        "surface": float(sum(branch.area.to_decimal(u.um ** 2) for branch in non_soma_branches)),
        "volume": float(sum(branch.volume.to_decimal(u.um ** 3) for branch in non_soma_branches)),
        "pathDistance": float(tree.max_path_distance_excluding_soma.to_decimal(u.um)),
        "eucDistance": float(tree.max_euclidean_distance_excluding_soma.to_decimal(u.um)),
        "soma_Surface": float(sum(branch.area.to_decimal(u.um ** 2) for branch in soma_branches)),
    }


def compute_neuromorpho_metrics(measurement):
    return {
        "n_stems": int(round(float(measurement["n_stems"]))),
        "n_branch": int(round(float(measurement["n_branch"]))),
        "n_bifs": int(round(float(measurement["n_bifs"]))),
        "length": float(measurement["length"]),
        "surface": float(measurement["surface"]),
        "volume": float(measurement["volume"]),
        "pathDistance": float(measurement["pathDistance"]),
        "eucDistance": float(measurement["eucDistance"]),
        "soma_Surface": float(measurement["soma_Surface"]),
    }


def compare_tree_with_measurement(tree, measurement):
    braincell_metrics = compute_braincell_metrics(tree)
    neuromorpho_metrics = compute_neuromorpho_metrics(measurement)
    metric_results = {}

    for key, target in neuromorpho_metrics.items():
        actual = braincell_metrics[key]
        if key in EXACT_MATCH_KEYS:
            abs_diff = abs(actual - target)
            rel_diff = None
            passed = actual == target
        else:
            abs_diff = abs(actual - target)
            if abs(target) <= 1e-12:
                rel_diff = 0.0 if abs_diff <= 1e-12 else None
                passed = abs_diff <= 1e-12
            else:
                rel_diff = abs_diff / abs(target)
                passed = rel_diff < FLOAT_THRESHOLDS[key]

        metric_results[key] = {
            "braincell": actual,
            "neuromorpho": target,
            "abs_diff": abs_diff,
            "rel_diff": rel_diff,
            "threshold": FLOAT_THRESHOLDS.get(key),
            "pass": passed,
        }

    non_soma_float_keys = [
        key for key in metric_results
        if key not in EXACT_MATCH_KEYS and key != "soma_Surface"
    ]
    non_soma_rel_diffs = [
        metric_results[key]["rel_diff"]
        for key in non_soma_float_keys
        if metric_results[key]["rel_diff"] is not None
    ]

    integer_pass = all(metric_results[key]["pass"] for key in EXACT_MATCH_KEYS)
    non_soma_float_pass = all(metric_results[key]["pass"] for key in non_soma_float_keys)
    soma_surface_pass = metric_results["soma_Surface"]["pass"]

    return {
        "braincell_metrics": braincell_metrics,
        "neuromorpho_metrics": neuromorpho_metrics,
        "metric_results": metric_results,
        "integer_pass": integer_pass,
        "non_soma_float_pass": non_soma_float_pass,
        "soma_surface_pass": soma_surface_pass,
        "overall_pass": integer_pass and non_soma_float_pass and soma_surface_pass,
        "non_soma_max_rel_diff": max(non_soma_rel_diffs) if non_soma_rel_diffs else 0.0,
        "soma_surface_rel_diff": metric_results["soma_Surface"]["rel_diff"],
    }
            


## 1. Batch query and cache neurons

Use `q` and `fq` as the NeuroMorpho query入口, then cache every matched neuron into:

`develop_doc/data/neuromorpho/<neuron_id>/`

Each neuron folder will contain:

- the standardized `SWC`, the original file, or both
- `metadata.json`
- the full NeuroMorpho measurement payload and links needed to trace the neuron later
            


In [ ]:
q = "species:mouse"
fq = [
    "brain_region:neocortex",
]
size = 20
page_start = 0
max_pages = 2
sort = "neuron_id,asc"
download_mode = "standard"  # choose from: "standard", "original", "both"
overwrite = False
output_root = repo_root / "develop_doc" / "data" / "neuromorpho"

neurons, query_urls = fetch_neuron_batch(
    q=q,
    fq=fq,
    size=size,
    page_start=page_start,
    max_pages=max_pages,
    sort=sort,
)
if not neurons:
    raise RuntimeError("No NeuroMorpho records matched the current filters. Relax q/fq and run again.")

print("q:", q)
print("fq:", fq)
print("size:", size)
print("page_start:", page_start)
print("max_pages:", max_pages)
print("download_mode:", download_mode)
print("output_root:", output_root)
print()
for query_url in query_urls:
    print("query_url:", query_url)
print()
print("matched_neurons:", len(neurons))
print()
for neuron in neurons:
    print(f"id={neuron['neuron_id']} name={neuron['neuron_name']}")
    print("  archive:", neuron.get("archive"))
    print("  brain_region:", neuron.get("brain_region"))
    print("  original_format:", neuron.get("original_format"))
    print()
            


In [ ]:
download_records = []
for neuron in neurons:
    record = cache_neuron_bundle(
        neuron,
        output_root,
        mode=download_mode,
        overwrite=overwrite,
    )
    download_records.append(record)

    print(f"cached neuron_id={neuron['neuron_id']} name={neuron['neuron_name']}")
    print("  folder:", record["folder"])
    print("  metadata:", record["metadata_path"])
    for item in record["download_items"]:
        print(f"  {item['kind']}: {item['filename']}")
        print("    url:", item["url"])
        print("    path:", item["path"])
        print("    downloaded_now:", item["downloaded_now"])
    print()

if download_records:
    sample_metadata = load_cached_metadata(download_records[0]["folder"])
    print("sample_metadata_file:", download_records[0]["metadata_path"])
    print("sample_measurement_url:", sample_metadata["measurement_url"])
    print("sample_measurement:")
    print(sample_metadata["measurement"])
            


## 2. Scan cached folders and batch compare

This step does not re-query NeuroMorpho. It only scans local folders under `develop_doc/data/neuromorpho/`, loads each cached standardized `SWC`, and compares it against the cached `measurement` stored in `metadata.json`.
            


In [ ]:
if output_root.exists():
    cache_dirs = sorted(path for path in output_root.iterdir() if path.is_dir())
else:
    cache_dirs = []

comparison_runs = []
summary = {
    "scanned": 0,
    "compared": 0,
    "skipped": 0,
    "passed": 0,
    "failed": 0,
    "errors": 0,
}

if not cache_dirs:
    print("No cached neuron folders were found under", output_root)
else:
    for folder in cache_dirs:
        summary["scanned"] += 1
        metadata_path = folder / "metadata.json"
        if not metadata_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: missing metadata.json")
            continue

        try:
            metadata = load_cached_metadata(folder)
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to read metadata.json: {exc}")
            continue

        measurement = metadata.get("measurement")
        if not measurement:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: metadata.json does not contain measurement")
            continue

        swc_path = find_standard_swc(folder, metadata)
        if swc_path is None or not swc_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no standardized SWC found")
            continue

        try:
            tree, report = Morpho.from_swc(swc_path, return_report=True, mode="neuromorpho")
            comparison = compare_tree_with_measurement(tree, measurement)
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to compare {swc_path}: {exc}")
            continue

        summary["compared"] += 1
        if comparison["overall_pass"]:
            summary["passed"] += 1
        else:
            summary["failed"] += 1

        comparison_runs.append(
            {
                "folder": str(folder),
                "neuron_id": metadata.get("neuron_id"),
                "neuron_name": metadata.get("neuron_name"),
                "swc_path": str(swc_path),
                "comparison": comparison,
            }
        )

        print("=" * 80)
        print(f"neuron_id: {metadata.get('neuron_id')}")
        print(f"neuron_name: {metadata.get('neuron_name')}")
        print(f"folder: {folder}")
        print(f"swc_path: {swc_path}")
        print()

        for key, result in comparison["metric_results"].items():
            rel_text = "n/a" if result["rel_diff"] is None else f"{result['rel_diff']:.6%}"
            print(
                f"{key}: braincell={result['braincell']:.6f}, neuromorpho={result['neuromorpho']:.6f}, "
                f"abs_diff={result['abs_diff']:.6f}, rel_diff={rel_text}, pass={result['pass']}"
            )

        print()
        print(f"integer_pass: {comparison['integer_pass']}")
        print(f"non_soma_max_rel_diff: {comparison['non_soma_max_rel_diff']:.6%}")
        print(
            "soma_surface_rel_diff:",
            "n/a" if comparison["soma_surface_rel_diff"] is None else f"{comparison['soma_surface_rel_diff']:.6%}",
        )
        print(f"non_soma_float_pass: {comparison['non_soma_float_pass']}")
        print(f"soma_surface_pass: {comparison['soma_surface_pass']}")
        print(f"overall_pass: {comparison['overall_pass']}")

    print("=" * 80)
    print("scan summary")
    print("scanned:", summary["scanned"])
    print("compared:", summary["compared"])
    print("skipped:", summary["skipped"])
    print("passed:", summary["passed"])
    print("failed:", summary["failed"])
    print("errors:", summary["errors"])

comparison_runs
            
